In [ ]:
import pandas as pd
import numpy as np
import pickle
from typing import Dict, Tuple
# Unlike the LI notebook query, this intentionally does NOT filter to PA-ending events.

query = """
WITH startTimes AS (
    SELECT
        gameUID,
        localDateTime
    FROM [base].[mitts_trackman_play]
    WHERE pitchNo = 1
    GROUP BY gameUID, localDateTime),
endTimes AS (
    SELECT
        gameUID,
        MAX(localDateTime) AS localDateTime
    FROM [base].[mitts_trackman_play]
    GROUP BY gameUID)
SELECT
    PLAY.playID,
    PLAY.gameUID,
    PLAY.localDateTime,
    startTimes.localDateTime AS startTime,
    endTimes.localDateTime AS endTime,
    CASE
        WHEN DATEPART(HOUR, PLAY.localDateTime) < 5
        THEN DATEADD(DAY, -1, PLAY.localDateTime)
        ELSE PLAY.localDateTime
    END AS dateNew,
    EBE.date,
    PLAY.batterTeam,
    PLAY.pitcherTeam,
    pitchCall,
    playResult,
    kOrBB,
    balls,
    strikes,
    outs,
    plateLocHeight,
    plateLocSide,
    in_zone,
    zone_loc,
    hit_type_classification,
    in_play,
    r_1B,
    r_2B,
    r_3B,
    PLAY.inning,
    PLAY.PAofInning,
    EBE.PAofInning AS EBE_PAofInning,
    PLAY.topBottom,
    outsOnPlay,
    runsScored,
    batterName,
    pitcherName,
    pitchofPA,
    max_event_of_PA,
    event_of_PA,
    r1_name,
    r2_name,
    r3_name,
    description,
    SESS.leagueShortName
FROM [base].[mitts_trackman_play] PLAY
JOIN [base].[mitts_trackman_pitch] PITCH
    ON PITCH.playID = PLAY.playID
JOIN [base].[mitts_trackman_session] SESS
    ON SESS.gameUID = PLAY.gameUID
JOIN [base].[vw_mitts_pbp_outcomes] pbp
    ON pbp.playID = PLAY.playID
JOIN startTimes
    ON startTimes.gameUID = PLAY.gameUID
JOIN endTimes
    ON endTimes.gameUID = PLAY.gameUID
JOIN [base].[scraped_baserunner_ebe] EBE
    ON PLAY.batterTeam = EBE.batterTeam
    AND PLAY.pitcherTeam = EBE.pitcherTeam
    AND PLAY.inning = EBE.inning
    AND PLAY.PAofInning = EBE.PAofInning
    AND PLAY.topBottom = EBE.topBottom
    AND YEAR(EBE.date) = YEAR(PLAY.localDateTime)
    AND MONTH(EBE.date) = MONTH(PLAY.localDateTime)
    AND EBE.date = CASE
        WHEN CAST(PLAY.localDateTime AS DATE) != CAST(startTimes.localDateTime AS DATE)
        THEN CAST(startTimes.localDateTime AS DATE)
        ELSE CAST(PLAY.localDateTime AS DATE)
    END
WHERE YEAR(PLAY.localDateTime) IN (2023, 2024, 2025)
    AND PLAY.batterTeam IN (
        SELECT teamName FROM [base].[vw_mitts_power_5_plus])
    AND PLAY.pitcherTeam IN (
        SELECT teamName FROM [base].[vw_mitts_power_5_plus])
    AND PLAY.localDateTime >= startTimes.localDateTime
    AND PLAY.localDateTime <= endTimes.localDateTime
ORDER BY PLAY.gameUID, PLAY.localDateTime"""

# Create database connection and run query
#User and Password hidden from public view

pitches = db_conn.run_query(query, file=False)

print(f"Loaded {len(pitches):,} Trackman pitch rows")
print(f"Unique games: {pitches['gameUID'].nunique():,}")
print(f"Columns returned: {len(pitches.columns)}")

display(pitches.head())

In [ ]:
# Step 1: Join final PA-level Leverage Index onto Trackman pitch/play-by-play rows
PA_KEYS = ['gameUID', 'inning', 'topBottom', 'PAofInning']
ANALYSIS_YEAR = 2025

# Keep only last season's Trackman pitches
pitches_2025 = pitches.copy()
pitches_2025['year'] = pd.to_datetime(pitches_2025['localDateTime']).dt.year
pitches_2025 = pitches_2025.loc[pitches_2025['year'] == ANALYSIS_YEAR].copy()

# Load the final LI value for each plate appearance
li_pa = pd.read_parquet(#file path hidden from public)

# Confirm one LI value exists per PA before merging
assert not li_pa.duplicated(PA_KEYS).any(), "Duplicate PA rows found in li_by_pa."

# Add that PA's LI to every pitch within the PA
pitches_with_li = pitches_2025.merge(li_pa[PA_KEYS + ['leverage_index']],on=PA_KEYS, how='inner',validate='many_to_one')

print(f"Pitches before LI join: {len(pitches_2025):,}")
print(f"Pitches matched to LI: {len(pitches_with_li):,}")
print(f"Unique PAs matched: {pitches_with_li[PA_KEYS].drop_duplicates().shape[0]:,}")

display(pitches_with_li[['gameUID', 'inning', 'topBottom', 'PAofInning','pitchofPA', 'pitcherName', 'batterName', 'leverage_index']].head(10))

In [ ]:
# Step 2: Configure and create LI buckets
# Change only these three shares. They must add to 1.00.
LI_BUCKET_SHARES = {'Low': 1/3, 'Medium': 1/3, 'High': 1/3,}

assert np.isclose(sum(LI_BUCKET_SHARES.values()), 1.0), ('LI_BUCKET_SHARES must add to 1.00.')
# Derive percentile cutoffs and labels from the configuration.
low_cutoff_pct = LI_BUCKET_SHARES['Low']
high_cutoff_pct = LI_BUCKET_SHARES['Low'] + LI_BUCKET_SHARES['Medium']
low_label = f"Low LI (bottom {LI_BUCKET_SHARES['Low']:.0%})"
medium_label = f"Medium LI ({low_cutoff_pct:.0%}-{high_cutoff_pct:.0%})"
high_label = f"High LI (top {LI_BUCKET_SHARES['High']:.0%})"
bucket_order = [low_label, medium_label, high_label]

# One LI observation per PA prevents longer PAs from affecting bucket cutoffs.
pa_li = (pitches_with_li[PA_KEYS + ['leverage_index']].drop_duplicates(PA_KEYS).copy())
p_low = pa_li['leverage_index'].quantile(low_cutoff_pct)
p_high = pa_li['leverage_index'].quantile(high_cutoff_pct)

pa_li['li_bucket'] = np.select([pa_li['leverage_index'] < p_low,pa_li['leverage_index'] > p_high,],[ low_label, high_label,],default=medium_label)
# Add the PA bucket back to every pitch within that PA.
pitches_with_li = (pitches_with_li.drop(columns='leverage_index').merge(pa_li, on=PA_KEYS, how='left', validate='many_to_one'))

bucket_summary = (pa_li.groupby('li_bucket', observed=False)
                  .agg(PAs=('leverage_index', 'size'),avg_LI=('leverage_index', 'mean'),min_LI=('leverage_index', 'min'),max_LI=('leverage_index', 'max'),).reindex(bucket_order))
bucket_summary['PA_share'] = 100 * bucket_summary['PAs'] / bucket_summary['PAs'].sum()
print(f'Low/medium cutoff LI: {p_low:.3f}')
print(f'Medium/high cutoff LI: {p_high:.3f}')
display(bucket_summary)

In [ ]:
# Step 3: Keep the final event for each plate appearance
# This will be used for PA-based stats: K%, BB%, wOBA, and run value.

pa_events = ( pitches_with_li .loc[pitches_with_li['event_of_PA'].eq(pitches_with_li['max_event_of_PA'])]
    .sort_values(PA_KEYS + ['localDateTime', 'pitchofPA'])
    .drop_duplicates(PA_KEYS, keep='last')
    .copy())

print(f'Unique PAs in LI data: {pitches_with_li[PA_KEYS].drop_duplicates().shape[0]:,}')
print(f'PA-ending events retained: {len(pa_events):,}')
display(pa_events[ PA_KEYS + ['pitcherName', 'batterName', 'li_bucket','pitchCall', 'playResult', 'kOrBB','runsScored', 'description' ]].head(15))

In [ ]:
# Step 4: Pitch-level KPIs by pitcher and leverage bucket:Strike%, Heart%, Whiff%, CSW%, and Chase%
# Remove administrative/non-pitch records
pitch_data = pitches_with_li.loc[pitches_with_li['pitchCall'].notna() & pitches_with_li['pitchCall'].ne('undefined')].copy()
# Trackman pitch-call groups
strike_calls = [ 'strikecalled', 'strikeswinging', 'foulballnotfieldable', 'foulballfieldable', 'inplay']
swing_calls = ['strikeswinging','foulballnotfieldable','foulballfieldable','inplay']

# Create pitch-level flags
pitch_data['is_strike'] = pitch_data['pitchCall'].isin(strike_calls)
pitch_data['is_swing'] = pitch_data['pitchCall'].isin(swing_calls)
pitch_data['is_whiff'] = pitch_data['pitchCall'].eq('strikeswinging')
# Called Strike + Whiff
pitch_data['is_csw'] = pitch_data['pitchCall'].isin( ['strikecalled', 'strikeswinging'])
# Heart-zone pitches
pitch_data['is_heart'] = pitch_data['zone_loc'].eq('heart')
# Chase pitches and swings at chase pitches
pitch_data['is_chase_pitch'] = pitch_data['zone_loc'].eq('chase')
pitch_data['is_chase_swing'] = (pitch_data['is_chase_pitch'] & pitch_data['is_swing'])

# Summarize by pitcher, team, and LI bucket
pitch_kpis = ( pitch_data
.groupby(['pitcherName', 'pitcherTeam', 'li_bucket'], as_index=False).agg(pitches=('pitchCall', 'size'), swings=('is_swing', 'sum'),strikes=('is_strike', 'sum'),
        whiffs=('is_whiff', 'sum'),csw=('is_csw', 'sum'),heart_pitches=('is_heart', 'sum'),chase_pitches=('is_chase_pitch', 'sum'),chase_swings=('is_chase_swing', 'sum')))
# Calculate rates
pitch_kpis['strike_pct'] = (100 * pitch_kpis['strikes'] / pitch_kpis['pitches'])
pitch_kpis['heart_pct'] = (100 * pitch_kpis['heart_pitches'] / pitch_kpis['pitches'])
# Whiff% = whiffs / swings
pitch_kpis['whiff_pct'] = (100 * pitch_kpis['whiffs'] / pitch_kpis['swings'])
pitch_kpis['csw_pct'] = (100 * pitch_kpis['csw'] / pitch_kpis['pitches'])
# Chase% = swings at chase pitches / all chase pitches
pitch_kpis['chase_pct'] = (100 * pitch_kpis['chase_swings'] / pitch_kpis['chase_pitches'])
# Order the buckets logically instead of alphabetically
pitch_kpis['li_bucket'] = pd.Categorical( pitch_kpis['li_bucket'],categories=bucket_order,ordered=True)
pitch_kpis = pitch_kpis.sort_values(['li_bucket', 'pitches'],ascending=[True, False])
# Display every pitcher/bucket row in one sortable table
display( pitch_kpis .reset_index(drop=True))

In [ ]:
# Step 5: PA-level KPIs by pitcher and leverage bucket
# Metrics: K%, BB%, and ICR
# Remove runner-only events; these are not batter plate appearances. Examples: caught stealing or stolen base.
pa_data = pa_events.loc[~pa_events['playResult'].isin(['caughtstealing', 'stolenbase'])].copy()
# K% and BB% flags from the exact kOrBB labels in your data
pa_data['is_strikeout'] = pa_data['kOrBB'].eq('strikeout')
pa_data['is_walk'] = pa_data['kOrBB'].eq('walk')

# Balls in play and ideal-contact balls in play
ideal_contact_types = [ 'Flare-or-Burner', 'Solid-Contact', 'Barrel']
pa_data['is_bip'] = pa_data['in_play'].eq(1)
pa_data['is_ideal_contact'] = ( pa_data['is_bip'] & pa_data['hit_type_classification'].isin(ideal_contact_types))

# Summarize by pitcher and leverage bucket
pa_kpis = ( pa_data .groupby(['pitcherName', 'pitcherTeam', 'li_bucket'], as_index=False)
    .agg(plate_appearances=('playID', 'size'),strikeouts=('is_strikeout', 'sum'),walks=('is_walk', 'sum'),balls_in_play=('is_bip', 'sum'),ideal_contact=('is_ideal_contact', 'sum')))
# Calculate the rates
pa_kpis['k_pct'] = ( 100 * pa_kpis['strikeouts'] / pa_kpis['plate_appearances'])
pa_kpis['bb_pct'] = (100 * pa_kpis['walks'] / pa_kpis['plate_appearances'])
pa_kpis['icr_pct'] = ( 100 * pa_kpis['ideal_contact'] / pa_kpis['balls_in_play'])
# Keep the same Low → Medium → High ordering as the pitch-level table
pa_kpis['li_bucket'] = pd.Categorical(pa_kpis['li_bucket'],categories=bucket_order,ordered=True)
pa_kpis = pa_kpis.sort_values( ['li_bucket', 'plate_appearances'],ascending=[True, False])
# Display every pitcher/bucket row in one sortable table
display(pa_kpis.sort_values( ['li_bucket', 'plate_appearances'], ascending=[True, False]).reset_index(drop=True))

In [ ]:
# Step 6: Calculate pitcher Run Value and Run Value per 100 pitches
# Positive values = good for the pitcher / bad for the offense.

run_value_lookup = pd.read_csv( #hidden file path')
# Confirm that every lookup result/count/outs combination is unique.
assert not run_value_lookup.duplicated(['result', 'count', 'outs']).any(), 'Duplicate rows found in the run-value lookup.'
# Use the cleaned pitch-level data from the pitch KPI cell.
pitch_rv = pitch_data.copy()
# Create the pre-pitch count in the same format as the CSV: e.g., "2-1".
pitch_rv['count'] = (pd.to_numeric(pitch_rv['balls'], errors='coerce').astype('Int64').astype(str)+ '-'+ pd.to_numeric(pitch_rv['strikes'], errors='coerce').astype('Int64').astype(str))
pitch_rv['outs'] = pd.to_numeric(pitch_rv['outs'],errors='coerce').astype('Int64')

# Map Trackman pitch calls to the run-value file's result names.
pitch_call_to_rv_result = {'ballcalled': 'ball','ballindirt': 'ball','ballintentional': 'ball','strikecalled': 'called_strike','strikeswinging': 'swinging_strike','foulball': 'foul',
        'foulballnotfieldable': 'foul','foulballfieldable': 'foul','hitbypitch': 'hit_by_pitch'}
pitch_rv['rv_result'] = pitch_rv['pitchCall'].map(pitch_call_to_rv_result)

# For balls put in play, use the recorded play result.
inplay_to_rv_result = {'single': 'single','double': 'double','triple': 'triple','homerun': 'home_run','out': 'field_out','fielderschoice': 'field_out','sacrifice': 'field_out','error': 'error'}
is_inplay = pitch_rv['pitchCall'].eq('inplay')
pitch_rv.loc[is_inplay, 'rv_result'] = ( pitch_rv.loc[is_inplay, 'playResult'] .map(inplay_to_rv_result))
# Join the batter-perspective delta in run expectancy from the lookup.
pitch_rv = pitch_rv.merge(run_value_lookup.rename(columns={ 'result': 'rv_result', 'deltaRunExpectancy': 'batter_run_value'}),
    on=['rv_result', 'count', 'outs'],how='left',validate='many_to_one')
# Reverse the sign: a negative batter value is a positive pitcher value.
pitch_rv['pitcher_run_value'] = -pitch_rv['batter_run_value']
unmatched_rv = pitch_rv.loc[pitch_rv['pitcher_run_value'].isna()].copy()

print(f'RV lookup matched: {len(pitch_rv) - len(unmatched_rv):,} ' f'of {len(pitch_rv):,} pitches ' f'({1 - len(unmatched_rv) / len(pitch_rv):.3%})')
if len(unmatched_rv) > 0:
    display(unmatched_rv[ ['pitchCall', 'playResult', 'count', 'outs', 'rv_result']].fillna('MISSING').value_counts(dropna=False).reset_index(name='unmatched_pitches'))

# Aggregate RV by pitcher and leverage bucket.
run_value_kpis = (pitch_rv.dropna(subset=['pitcher_run_value']).groupby(['pitcherName', 'pitcherTeam', 'li_bucket'], as_index=False)
    .agg(rv_pitches=('pitcher_run_value', 'size'),run_value=('pitcher_run_value', 'sum')))
# RV per 100 pitches.
run_value_kpis['run_value_per_100'] = (100* run_value_kpis['run_value']/ run_value_kpis['rv_pitches'])
run_value_kpis['li_bucket'] = pd.Categorical( run_value_kpis['li_bucket'],categories=bucket_order,ordered=True)
display(run_value_kpis.sort_values( ['li_bucket', 'run_value_per_100'], ascending=[True, False]).reset_index(drop=True))

In [ ]:
# Step 7: Calculate wOBA by pitcher and leverage bucket
# Uses the outcome weights provided:
# BB = 0.804, HBP = 0.823, 1B = 0.936, 2B = 1.257, 3B = 1.495, HR = 1.809
woba_data = pa_data.copy()

# Mark each weighted offensive outcome on the PA-ending event
woba_data['woba_bb'] = woba_data['kOrBB'].eq('walk')
woba_data['woba_hbp'] = woba_data['pitchCall'].eq('hitbypitch')
woba_data['woba_1b'] = woba_data['playResult'].eq('single')
woba_data['woba_2b'] = woba_data['playResult'].eq('double')
woba_data['woba_3b'] = woba_data['playResult'].eq('triple')
woba_data['woba_hr'] = woba_data['playResult'].eq('homerun')

# Weighted numerator for every PA; all other outcomes contribute zero.
woba_data['woba_value'] = (
      0.804 * woba_data['woba_bb']
    + 0.823 * woba_data['woba_hbp']
    + 0.936 * woba_data['woba_1b']
    + 1.257 * woba_data['woba_2b']
    + 1.495 * woba_data['woba_3b']
    + 1.809 * woba_data['woba_hr'])

woba_kpis = ( woba_data.groupby(['pitcherName', 'pitcherTeam', 'li_bucket'], as_index=False)
    .agg( woba_plate_appearances=('playID', 'size'), woba_numerator=('woba_value', 'sum')))

# Lower is better for pitchers.
woba_kpis['woba'] = (woba_kpis['woba_numerator'] / woba_kpis['woba_plate_appearances'])
woba_kpis['li_bucket'] = pd.Categorical(woba_kpis['li_bucket'],categories=bucket_order,ordered=True)
display( woba_kpis.sort_values(['li_bucket', 'woba'], ascending=[True, True]).reset_index(drop=True))

In [ ]:
# Step 8: Combine all pitcher KPIs by leverage bucket
# This cell rebuilds all_kpis from scratch every time it runs.

all_kpis = (pitch_kpis[['pitcherName', 'pitcherTeam', 'li_bucket', 'pitches', 'swings', 'chase_pitches', 'strike_pct', 'heart_pct', 'whiff_pct', 'csw_pct', 'chase_pct']]
    .merge(pa_kpis[ ['pitcherName', 'pitcherTeam', 'li_bucket','plate_appearances', 'balls_in_play', 'k_pct', 'bb_pct','icr_pct']],
        on=['pitcherName', 'pitcherTeam', 'li_bucket'],how='inner',validate='one_to_one')
    .merge(run_value_kpis[['pitcherName', 'pitcherTeam', 'li_bucket', 'rv_pitches', 'run_value', 'run_value_per_100']],
        on=['pitcherName', 'pitcherTeam', 'li_bucket'],how='left',validate='one_to_one')
    .merge(woba_kpis[ ['pitcherName', 'pitcherTeam', 'li_bucket', 'woba_plate_appearances', 'woba']],
        on=['pitcherName', 'pitcherTeam', 'li_bucket'],how='left',validate='one_to_one'))

all_kpis = ( all_kpis.sort_values(['li_bucket', 'woba'], ascending=[True, True]).reset_index(drop=True))
display(all_kpis)

In [ ]:
# Step 9: Create a qualified KPI table for charts and leaderboards

MIN_PITCHES = 50
MIN_PA = 15
MIN_BIP = 10
qualified_kpis = (all_kpis.loc[ (all_kpis['pitches'] >= MIN_PITCHES) & (all_kpis['plate_appearances'] >= MIN_PA) & (all_kpis['balls_in_play'] >= MIN_BIP)].copy())

print(f'Rows before filtering: {len(all_kpis):,}')
print(f'Rows after filtering: {len(qualified_kpis):,}')
display(qualified_kpis .sort_values(['li_bucket', 'run_value_per_100'], ascending=[True, False]).reset_index(drop=True))

In [ ]:
import matplotlib.pyplot as plt

# ---------- Easy-to-edit selection ----------
CLEMSON_TEAM = 'TEAM'
PITCHER_NAME = 'NAME'
# -------------------------------------------
short_bucket_labels = ['Low', 'Medium', 'High']
bucket_colors = ['#6C757D', '#F66733', '#522D80']

# Pull the selected pitcher's leverage rows
selected_rows = all_kpis.query('pitcherTeam == @TEAM and pitcherName == @PITCHER_NAME').copy()

if selected_rows.empty:
    raise ValueError( f"No rows found for {PITCHER_NAME} on {CLEMSON_TEAM}. " "Check the spelling against " "all_kpis[['pitcherName', 'pitcherTeam']].drop_duplicates().")

pitcher_profile = (selected_rows.set_index('li_bucket').reindex(bucket_order))

# Build weighted team averages for the selected pitcher's team.
team_rows = all_kpis.loc[all_kpis['pitcherTeam'].eq(CLEMSON_TEAM)].copy()
team_rows['k_num'] = team_rows['k_pct'] / 100 * team_rows['plate_appearances']
team_rows['bb_num'] = team_rows['bb_pct'] / 100 * team_rows['plate_appearances']
team_rows['icr_num'] = team_rows['icr_pct'] / 100 * team_rows['balls_in_play']
team_rows['strike_num'] = team_rows['strike_pct'] / 100 * team_rows['pitches']
team_rows['heart_num'] = team_rows['heart_pct'] / 100 * team_rows['pitches']
team_rows['whiff_num'] = team_rows['whiff_pct'] / 100 * team_rows['swings']
team_rows['csw_num'] = team_rows['csw_pct'] / 100 * team_rows['pitches']
team_rows['chase_num'] = team_rows['chase_pct'] / 100 * team_rows['chase_pitches']
team_rows['woba_num'] = team_rows['woba'] * team_rows['woba_plate_appearances']

team_profile = (team_rows.groupby('li_bucket', observed=False, as_index=True)
    .agg(pitches=('pitches', 'sum'),swings=('swings', 'sum'),chase_pitches=('chase_pitches', 'sum'),plate_appearances=('plate_appearances', 'sum'),
        balls_in_play=('balls_in_play', 'sum'), rv_pitches=('rv_pitches', 'sum'), run_value=('run_value', 'sum'), woba_plate_appearances=('woba_plate_appearances', 'sum'),
         k_num=('k_num', 'sum'), bb_num=('bb_num', 'sum'), icr_num=('icr_num', 'sum'), strike_num=('strike_num', 'sum'), heart_num=('heart_num', 'sum'),
        whiff_num=('whiff_num', 'sum'),csw_num=('csw_num', 'sum'),chase_num=('chase_num', 'sum'),woba_num=('woba_num', 'sum') ) .reindex(bucket_order))

# Calculate weighted team rates for each leverage bucket.
team_profile['k_pct'] = 100 * team_profile['k_num'] / team_profile['plate_appearances']
team_profile['bb_pct'] = 100 * team_profile['bb_num'] / team_profile['plate_appearances']
team_profile['icr_pct'] = 100 * team_profile['icr_num'] / team_profile['balls_in_play']
team_profile['strike_pct'] = 100 * team_profile['strike_num'] / team_profile['pitches']
team_profile['heart_pct'] = 100 * team_profile['heart_num'] / team_profile['pitches']
team_profile['whiff_pct'] = 100 * team_profile['whiff_num'] / team_profile['swings']
team_profile['csw_pct'] = 100 * team_profile['csw_num'] / team_profile['pitches']
team_profile['chase_pct'] = 100 * team_profile['chase_num'] / team_profile['chase_pitches']
team_profile['woba'] = team_profile['woba_num'] / team_profile['woba_plate_appearances']
team_profile['run_value_per_100'] = (100 * team_profile['run_value'] / team_profile['rv_pitches'])

metrics = [ 'k_pct', 'bb_pct', 'icr_pct', 'strike_pct', 'heart_pct', 'whiff_pct', 'csw_pct', 'chase_pct', 'woba', 'run_value', 'run_value_per_100']
metric_labels = {'k_pct': 'K%','bb_pct': 'BB%','icr_pct': 'Ideal Contact Rate','strike_pct': 'Strike%','heart_pct': 'Heart%','whiff_pct': 'Whiff%','csw_pct': 
                 'CSW%','chase_pct': 'Chase%','woba': 'wOBA','run_value': 'Run Value','run_value_per_100': 'RV per 100 Pitches'}

percent_metrics = {'k_pct', 'bb_pct', 'icr_pct', 'strike_pct','heart_pct', 'whiff_pct', 'csw_pct', 'chase_pct'}
pitcher_positive_metrics = {'run_value', 'run_value_per_100'}

# Fixed axis ranges: identical for every pitcher chart.
fixed_y_limits = {'k_pct': (0, 45),'bb_pct': (0, 25),'icr_pct': (0, 60),'strike_pct': (45, 80),'heart_pct': (0, 35),'whiff_pct': (0, 50),
    'csw_pct': (0, 45),'chase_pct': (0, 45),'woba': (0.15, 0.50),'run_value': (-10, 35),'run_value_per_100': (-5, 5)}

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()
x_positions = range(len(bucket_order))

for ax, metric in zip(axes, metrics):
    actual_values = pd.to_numeric(pitcher_profile[metric],errors='coerce').to_numpy()
    # For total RV, the team reference is the expected RV at the
    # selected pitcher's number of RV-eligible pitches.
    if metric == 'run_value':
        reference_values = (team_profile['run_value_per_100'].to_numpy()* pitcher_profile['rv_pitches'].to_numpy()/ 100)
        reference_label = f'{CLEMSON_TEAM} expected RV'
    else:
        reference_values = pd.to_numeric(team_profile[metric],errors='coerce').to_numpy()
        reference_label = f'{CLEMSON_TEAM} average'
    # Dashed blue line = team average within the same leverage bucket.
    ax.plot( x_positions, reference_values,linestyle='--',color='#1F77B4',linewidth=1.8,marker='o',markersize=6,markerfacecolor='white',markeredgewidth=1.5,label=reference_label,zorder=1)
    # Solid black line / colored markers = selected pitcher.
    ax.plot(x_positions,actual_values,color='#222222',linewidth=1.7,zorder=2)
    ax.scatter(x_positions,actual_values,s=95,c=bucket_colors,edgecolors='black',linewidths=0.75,zorder=3)
    ax.set_ylim(*fixed_y_limits[metric])
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels(short_bucket_labels)
    ax.set_title(metric_labels[metric], fontweight='bold')
    ax.grid(axis='y', alpha=0.25)
    if metric in percent_metrics:
        ax.set_ylabel('Rate (%)')
    elif metric in pitcher_positive_metrics:
        ax.set_ylabel('Pitcher favorable →')
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    else:
        ax.set_ylabel('Lower is better →')

    for x, value in enumerate(actual_values):
        if pd.notna(value):
            if metric in percent_metrics:
                label = f'{value:.1f}%'
            elif metric == 'woba':
                label = f'{value:.3f}'
            else:
                label = f'{value:.2f}'

            ax.annotate(label,(x, value),xytext=(0, 8),textcoords='offset points',ha='center',fontsize=9)
# Hide the unused twelfth subplot.
axes[-1].axis('off')

# Use one figure-level legend.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center', ncol=1,  frameon=False, bbox_to_anchor=(0.5, -0.01))

sample_text = '  |  '.join(f"{label}: {int(row['pitches'])} P, "
    f"{int(row['plate_appearances'])} PA, "f"{int(row['balls_in_play'])} BIP"for label, (_, row) in zip(short_bucket_labels, pitcher_profile.iterrows()))

fig.suptitle(f'{PITCHER_NAME} — Leverage KPI Profile ({CLEMSON_TEAM})\n{sample_text}',fontsize=16,fontweight='bold',y=1.02)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

In [ ]:
from matplotlib.lines import Line2D
#
# ---------- Easy-to-edit selection ----------
TEAM_NAME = 'TEAM'
# -------------------------------------------
HIGH_LI_BUCKET = bucket_order[-1]
# Qualification thresholds
MIN_PITCHES = 50
MIN_PA = 15
MIN_BIP = 10
# Qualified High-LI pitchers for the selected team
high_li = (all_kpis.loc[(all_kpis['pitcherTeam'].eq(TEAM_NAME))& (all_kpis['li_bucket'].eq(HIGH_LI_BUCKET))& (all_kpis['pitches'] >= MIN_PITCHES)
        & (all_kpis['plate_appearances'] >= MIN_PA) & (all_kpis['balls_in_play'] >= MIN_BIP)].copy().dropna(subset=['woba', 'run_value_per_100', 'plate_appearances', 'k_pct']))
if high_li.empty:
    raise ValueError( f'No qualified High-LI pitchers found for {TEAM_NAME}. ' 'Check TEAM_NAME or the qualification thresholds.')

# Calculate weighted High-LI team reference values.
team_high_li = (all_kpis.loc[(all_kpis['pitcherTeam'].eq(TEAM_NAME))& (all_kpis['li_bucket'].eq(HIGH_LI_BUCKET))].copy().dropna(subset=['woba', 'woba_plate_appearances', 'run_value', 'rv_pitches']))
team_woba = ((team_high_li['woba'] * team_high_li['woba_plate_appearances']).sum()/ team_high_li['woba_plate_appearances'].sum())
team_rv_per_100 = ( 100 * team_high_li['run_value'].sum() / team_high_li['rv_pitches'].sum())
#Bubble size represents High-LI plate appearances.
pa_min = high_li['plate_appearances'].min()
pa_max = high_li['plate_appearances'].max()
def bubble_size(pa):
    if pa_max == pa_min:
        return 450
    return 180 + 720 * (pa - pa_min) / (pa_max - pa_min)

bubble_sizes = high_li['plate_appearances'].apply(bubble_size)
# Slightly taller figure leaves room for the external legend and guide.
fig, ax = plt.subplots(figsize=(13, 10))
# Bubble position and appearance:
# x-axis = wOBA allowed
# y-axis = RV/100
# size = High-LI PA
# color = High-LI K%
scatter = ax.scatter(high_li['woba'],high_li['run_value_per_100'],s=bubble_sizes,c=high_li['k_pct'],cmap='viridis',edgecolors='black',linewidths=0.9,alpha=0.85,zorder=3)
# Team-average reference lines.
ax.axvline( team_woba, color='#1F77B4', linestyle='--', linewidth=1.5, zorder=1)
ax.axhline( team_rv_per_100, color='#1F77B4', linestyle='--', linewidth=1.5, zorder=1)
# Label each pitcher.
for _, row in high_li.iterrows():
    ax.annotate( row['pitcherName'], (row['woba'], row['run_value_per_100']), xytext=(7, 6), textcoords='offset points', fontsize=9, fontweight='medium')

ax.set_xlabel('High-Leverage wOBA Allowed  ← Better', fontsize=12)
ax.set_ylabel('High-Leverage RV per 100 Pitches  → Better', fontsize=12)
ax.set_title( f'{TEAM_NAME} High-Leverage Pitcher Outcomes', fontsize=16, fontweight='bold', pad=14)
ax.grid(alpha=0.25, zorder=0)
# K% color scale, placed outside the chart at right.
colorbar = fig.colorbar(scatter, ax=ax, pad=0.02)
colorbar.ax.set_title('High LI K%\nBubble Color Scale',fontsize=10,fontweight='bold',pad=10)
# Simple external legend: reference line and a single bubble-size explanation.
legend_handles = [Line2D( [0], [0], color='#1F77B4', linestyle='--',linewidth=1.8, marker='o', markerfacecolor='white', markeredgewidth=1.5,
        label=( f'{TEAM_NAME} High-LI average: ' f'wOBA {team_woba:.3f} | RV/100 {team_rv_per_100:.2f}')),
    Line2D([0], [0],marker='o',color='w', markerfacecolor='lightgray', markeredgecolor='black', markersize=11, label='Bubble size = High-LI plate appearances' )]

# Place the legend beneath the chart so it cannot cover pitcher bubbles.
fig.legend( handles=legend_handles, loc='lower center', bbox_to_anchor=(0.45, 0.035), ncol=1, frameon=False, fontsize=10)
# Put the interpretation guide outside the chart as well.
fig.text(0.45,0.115,'Best outcome profile: low wOBA + high RV/100',ha='center',fontsize=11, fontweight='bold')

# Reserve bottom space for the external guide and legend.
fig.subplots_adjust( left=0.09, right=0.88, top=0.90, bottom=0.20)
plt.show()

In [ ]:
# ---------- Easy-to-edit selection ----------
TEAM_NAME = 'TEAM'
# -------------------------------------------
LOW_LI_BUCKET = bucket_order[0]
HIGH_LI_BUCKET = bucket_order[-1]
# Qualification thresholds
MIN_PITCHES = 50
MIN_PA = 15
MIN_BIP = 10

# Clemson brand colors:
low_color = '#F56600'
high_color = '#522D80'
connector_color = '#4A4A4A'
# Keep only qualified Low- and High-LI rows for the selected team.
leverage_data = (all_kpis.loc[(all_kpis['pitcherTeam'].eq(TEAM_NAME)) & (all_kpis['li_bucket'].isin([LOW_LI_BUCKET, HIGH_LI_BUCKET])) & (all_kpis['pitches'] >= MIN_PITCHES)
        & (all_kpis['plate_appearances'] >= MIN_PA) & (all_kpis['balls_in_play'] >= MIN_BIP)].copy().dropna(subset=['woba', 'plate_appearances']))
# Place each qualified pitcher on one row with Low- and High-LI wOBA.
woba_comparison = (leverage_data.pivot(index='pitcherName',columns='li_bucket',values='woba')
    .dropna(subset=[LOW_LI_BUCKET, HIGH_LI_BUCKET]).rename(columns={ LOW_LI_BUCKET: 'low_woba', HIGH_LI_BUCKET: 'high_woba'}))
if woba_comparison.empty:
    raise ValueError(f'No pitchers on {TEAM_NAME} qualify in both Low LI and High LI. ' 'Consider lowering the qualification thresholds if needed.')

# Negative change = improved wOBA in High LI.
# Positive change = worse wOBA in High LI.
woba_comparison['delta_woba'] = (woba_comparison['high_woba'] - woba_comparison['low_woba'])
# Best High-LI change at the top; largest decline at the bottom.
woba_comparison = ( woba_comparison .sort_values('delta_woba', ascending=True) .reset_index())
# Weighted team Low- and High-LI wOBA reference values.
team_leverage_data = (all_kpis.loc[(all_kpis['pitcherTeam'].eq(TEAM_NAME)) & (all_kpis['li_bucket'].isin([LOW_LI_BUCKET, HIGH_LI_BUCKET]))].copy().dropna(subset=['woba', 'woba_plate_appearances']))
team_low_data = team_leverage_data.loc[team_leverage_data['li_bucket'].eq(LOW_LI_BUCKET)]
team_high_data = team_leverage_data.loc[team_leverage_data['li_bucket'].eq(HIGH_LI_BUCKET)]
team_low_woba = ((team_low_data['woba'] * team_low_data['woba_plate_appearances']).sum() / team_low_data['woba_plate_appearances'].sum())
team_high_woba = ((team_high_data['woba'] * team_high_data['woba_plate_appearances']).sum()/ team_high_data['woba_plate_appearances'].sum())
# ----------------------------------------------------------------
# Dumbbell chart
# ----------------------------------------------------------------
fig_height = max(6.5, len(woba_comparison) * 0.65 + 2.8)
fig, ax = plt.subplots(figsize=(13, fig_height))
y_positions = range(len(woba_comparison))
for y, (_, row) in zip(y_positions, woba_comparison.iterrows()):
    # Connect the pitcher’s Low-LI and High-LI wOBA.
    ax.plot([row['low_woba'], row['high_woba']],[y, y],color=connector_color,linewidth=2.2,alpha=0.80,zorder=2)

    # Low LI endpoint — Clemson orange.
    ax.scatter( row['low_woba'], y, s=145, color=low_color, edgecolor='black', linewidth=0.9, zorder=4)
    # High LI endpoint — Clemson purple.
    ax.scatter(  row['high_woba'], y, s=145,color=high_color,edgecolor='black', linewidth=0.9, zorder=4)
# Staff reference lines stay understated behind the player comparison.
ax.axvline(team_low_woba,color=low_color, linestyle='--', linewidth=1.4, alpha=0.65, zorder=1)
ax.axvline(  team_high_woba,  color=high_color,  linestyle='--',  linewidth=1.4,  alpha=0.65,  zorder=1)
# Compact delta values live beside each pitcher name instead of on the chart.
y_labels = [ f"{row['pitcherName']}   Δ {row['delta_woba']:+.3f}" for _, row in woba_comparison.iterrows()]
x_min = min(woba_comparison[['low_woba', 'high_woba']].min()) - 0.015
x_max = max(woba_comparison[['low_woba', 'high_woba']].max()) + 0.015
ax.set_xlim(x_min, x_max)
ax.set_yticks(list(y_positions))
ax.set_yticklabels(y_labels, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('wOBA Allowed  ← Better', fontsize=12, labelpad=14)
ax.set_title( f'{TEAM_NAME} Low-to-High Leverage wOBA Change', fontsize=16, fontweight='bold', pad=15)
ax.text( 0.5, 1.01, 'Pitchers ordered by High-LI wOBA change', transform=ax.transAxes, ha='center', va='bottom', fontsize=10, color='#555555')
ax.grid(axis='x', alpha=0.22, zorder=0)
ax.grid(axis='y', alpha=0.10, zorder=0)
# External legend: no overlap with player dots.
legend_handles = [Line2D([0], [0],  marker='o',  color='w',  markerfacecolor=low_color,  markeredgecolor='black',  markersize=10,  label=f'Low LI wOBA | Team: {team_low_woba:.3f}'),
    Line2D(  [0], [0],  marker='o',  color='w',  markerfacecolor=high_color,  markeredgecolor='black',  markersize=10,  label=f'High LI wOBA | Team: {team_high_woba:.3f}'),
    Line2D( [0], [0], color=connector_color, linewidth=2.2, label='Δ wOBA = High LI minus Low LI')]
fig.legend( handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=3, frameon=False, fontsize=10)
# Clear explanatory note with ample spacing from the x-axis title and legend.
fig.text( 0.5, 0.105, 'Negative Δ = improved wOBA in High LI   |   Positive Δ = worse wOBA in High LI', ha='center', fontsize=10,  fontweight='bold')
fig.subplots_adjust( left=0.24, right=0.95, top=0.89, bottom=0.22)
plt.show()

In [ ]:
import seaborn as sns
# Editable sample-size filters
MIN_HIGH_LI_PITCHES = 50
MIN_HIGH_LI_PA = 10
MIN_HIGH_LI_BIP = 10
HIGH_LI_BUCKET = bucket_order[-1]
high_li_staff = (all_kpis.loc[(all_kpis['pitcherTeam'].eq(CLEMSON_TEAM))& (all_kpis['li_bucket'].eq(HIGH_LI_BUCKET)) & (all_kpis['pitches'] >= MIN_HIGH_LI_PITCHES)
                              & (all_kpis['plate_appearances'] >= MIN_HIGH_LI_PA)].copy())
heatmap_metrics = ['k_pct', 'bb_pct', 'icr_pct','strike_pct','heart_pct','whiff_pct','csw_pct','chase_pct']
heatmap_labels = ['K% ↑','BB% ↓','ICR% ↓','Strike% ↑','Heart% ↓','Whiff% ↑','CSW% ↑','Chase% ↑']
# Standardize each metric within Clemson's High-LI group.
# Reverse metrics where lower performance is better.
staff_scores = high_li_staff[heatmap_metrics].copy()
for metric in heatmap_metrics:
    metric_std = staff_scores[metric].std()
    if metric_std == 0 or pd.isna(metric_std):
        staff_scores[metric] = 0
    else:
        staff_scores[metric] = (staff_scores[metric] - staff_scores[metric].mean()) / metric_std

# Lower is better for these metrics, so reverse their colors/scores.
staff_scores[['bb_pct', 'icr_pct', 'heart_pct']] *= -1
staff_scores.index = ( high_li_staff['pitcherName'] + ' — ' + high_li_staff['pitches'].astype(int).astype(str) + ' pitches')
staff_scores.columns = heatmap_labels
# Sort from strongest to weakest overall current High-LI profile
staff_scores = staff_scores.loc[staff_scores.mean(axis=1).sort_values(ascending=False).index]
# Actual values are printed in each heatmap cell
annotations = high_li_staff.set_index(high_li_staff['pitcherName']+ ' — '+ high_li_staff['pitches'].astype(int).astype(str)+ ' pitches')[heatmap_metrics].loc[staff_scores.index]
annotations.columns = heatmap_labels
annotations = annotations.map(lambda x: f'{x:.1f}%')
plt.figure(figsize=(14, max(6, len(staff_scores) * 0.45)))
sns.heatmap(staff_scores,annot=annotations,fmt='',cmap='RdYlGn',center=0,linewidths=0.5,cbar_kws={'label': 'Relative High-Leverage Performance Within Clemson'})
plt.title( f'Clemson High-Leverage KPI Profile\n' f'Minimum: {MIN_HIGH_LI_PITCHES} pitches and {MIN_HIGH_LI_PA} PA', fontweight='bold', pad=14)
plt.xlabel('')
plt.ylabel('')
plt.tight_layout()
plt.show()